In [ ]:
import pandas as pd
import numpy as np
import glob
import networkx as nx
import os
import matplotlib.pyplot as plt

# Configuration: minimum profit threshold (e.g., 1.0001 = 0.01% profit after fees/spread)
MIN_PROFIT_MULTIPLIER = 1.0001

# Typical forex spread in basis points (1 bp = 0.01%)
# Major pairs: 1-3 bps, minor: 5-10 bps, exotic: 20-50+ bps
# We use a conservative 5 bps (0.05%) per leg for all pairs
SPREAD_BPS = 5
SPREAD_FRACTION = SPREAD_BPS / 10000  # 0.0005 = 0.05%

# ────────────────────────────────────────────────────────────────
# Arbitrage Detector with Graph Topology Caching
# ────────────────────────────────────────────────────────────────
class ArbitrageDetector:
    """
    Arbitrage detector with cached graph topology.

    Optimization:
    - Graph structure (nodes, edges) cached and reused across timestamps
    - Only edge weights recomputed per timestamp
    - Uses the exact same per-source Bellman-Ford logic as original
    - Deduplicates edges like NetworkX (last pair in column order wins)
    """

    def __init__(self, currencies, spread_fraction=SPREAD_FRACTION, min_profit=MIN_PROFIT_MULTIPLIER):
        self.currencies = sorted(currencies)
        self.spread_fraction = spread_fraction
        self.min_profit = min_profit

        # Cached graph topology (built once per day)
        self._nodes = None
        self._edges = None           # List of (u, v, pair, formula) tuples - deduplicated
        
    def _build_topology(self, df_pivot):
        """Build graph topology once from all pairs present in the day's data.
        
        Mimics NetworkX's behavior: for each (u,v) direction, only the last
        pair in column order creates an edge (overwrites previous).
        """
        if self._nodes is not None:
            return  # Topology unchanged across timestamps

        pairs = [c for c in df_pivot.columns if not df_pivot[c].isna().all()]
        nodes = set()
        # Use dict to deduplicate: (u, v) -> (pair, formula), last wins
        edge_map = {}

        for pair in pairs:
            base, quote = pair[:3], pair[3:]
            nodes.add(base)
            nodes.add(quote)
            # base -> quote: sell base, buy quote at the ASK price of the pair
            edge_map[(base, quote)] = (pair, 'ask')
            # quote -> base: sell quote, buy base at 1 / BID of the pair
            edge_map[(quote, base)] = (pair, 'inv_bid')

        self._nodes = sorted(nodes)
        # Convert back to list of tuples (u, v, pair, formula)
        self._edges = [(u, v, pair, formula) for (u, v), (pair, formula) in edge_map.items()]
        
    def _update_weights(self, row):
        """Compute edge weights for one timestamp from midpoint rates."""
        weights = {}
        for u, v, pair, formula in self._edges:
            close = row.get(pair, np.nan)
            if pd.isna(close) or close <= 0:
                weights[(u, v)] = np.inf
                continue

            if formula == 'ask':
                price = close * (1 + self.spread_fraction / 2)
            else:  # inv_bid
                price = 1 / (close * (1 - self.spread_fraction / 2))
            weights[(u, v)] = -np.log(price)
        return weights
    
    def _normalize_cycle(self, cycle):
        """Normalize cycle to canonical form (start from minimum currency)."""
        cycle = cycle[:-1]  # drop the closing duplicate
        min_idx = min(range(len(cycle)), key=lambda i: cycle[i])
        return tuple(cycle[min_idx:] + cycle[:min_idx])
    
    def _find_all_negative_cycles(self, weights):
        """
        Find all negative cycles using per-source Bellman-Ford.
        Exact same logic as original find_all_negative_cycles function.
        """
        all_cycles = set()
        cycles_info = []
        nodes = self._nodes
        
        for source in nodes:
            distance = {node: float('inf') for node in nodes}
            predecessor = {node: None for node in nodes}
            distance[source] = 0
            
            # Bellman-Ford algorithm (V-1 relaxations)
            for _ in range(len(nodes) - 1):
                for u, v, pair, formula in self._edges:
                    w = weights.get((u, v), np.inf)
                    if w == np.inf:
                        continue
                    if distance[u] + w < distance[v]:
                        distance[v] = distance[u] + w
                        predecessor[v] = u
            
            # Check for negative cycles (Vth iteration)
            for u, v, pair, formula in self._edges:
                w = weights.get((u, v), np.inf)
                if w == np.inf:
                    continue
                if distance[u] + w < distance[v]:
                    curr = v
                    for _ in range(len(nodes)):
                        curr = predecessor.get(curr)
                        if curr is None:
                            break
                    if curr is None:
                        continue

                    cycle = []
                    visited = set()
                    start = curr
                    while True:
                        if curr is None or curr in visited:
                            break
                        visited.add(curr)
                        cycle.append(curr)
                        curr = predecessor.get(curr)
                        if curr == start and len(cycle) > 1:
                            cycle.append(curr)
                            break

                    if len(cycle) > 1 and curr == start:
                        cycle = cycle[::-1]
                        norm = self._normalize_cycle(cycle)
                        if norm not in all_cycles:
                            profit_log = sum(weights[(cycle[i], cycle[i+1])] 
                                           for i in range(len(cycle)-1))
                            profit = np.exp(-profit_log)
                            if profit >= self.min_profit:
                                all_cycles.add(norm)
                                cycles_info.append({
                                    "cycle": " → ".join(cycle),
                                    "log_sum": profit_log,
                                    "profit_multiplier": profit
                                })
        return cycles_info
    
    def detect(self, df_pivot):
        """Detect arbitrage cycles for all timestamps in a day's pivot table."""
        self._build_topology(df_pivot)
        
        all_cycles = []
        for timestamp, row in df_pivot.iterrows():
            weights = self._update_weights(row)
            cycles = self._find_all_negative_cycles(weights)
            for c in cycles:
                c["timestamp"] = timestamp
            all_cycles.extend(cycles)
        return all_cycles

In [12]:
def load_forex_files_by_day(folder_pattern="./Data/forex_intraday_*.csv"):
    files = sorted(glob.glob(folder_pattern))
    day_data = {}

    for file in files:
        df = pd.read_csv(file, parse_dates=["timestamp"])
        if df.empty:
            continue
        date = pd.to_datetime(df["timestamp"].iloc[0]).date()
        day_data[date] = df

    return day_data

all_days_data = load_forex_files_by_day()

In [ ]:
def build_graphs(df):
    """
    Build graphs with realistic bid/ask spread (legacy function kept for compatibility).
    
    Polygon's aggregates endpoint returns midpoint/close prices.
    We simulate bid/ask by applying a symmetric spread around the midpoint:
    - Ask = close * (1 + spread/2)  -> you pay more when buying base currency
    - Bid = close * (1 - spread/2)  -> you receive less when selling base currency
    
    For edge A -> B (sell A, buy B): use ASK price for A/B
    For edge B -> A (sell B, buy A): use 1 / BID price for A/B = 1 / (close * (1 - spread/2))
    """
    df = df[["timestamp", "pair", "close"]]
    df_pivot = df.pivot(index="timestamp", columns="pair", values="close")
    
    graphs = {}
    for timestamp, row in df_pivot.iterrows():
        G = nx.DiGraph()
        for pair, close in row.items():
            if pd.notna(close) and close > 0:
                from_currency = pair[:3]
                to_currency = pair[3:]
                
                # Simulate spread: ask = close * (1 + spread/2), bid = close * (1 - spread/2)
                ask_price = close * (1 + SPREAD_FRACTION / 2)
                bid_price = close * (1 - SPREAD_FRACTION / 2)
                
                # Edge: from_currency -> to_currency (sell from_currency, buy to_currency)
                # We pay the ASK price: we get less to_currency per from_currency
                G.add_edge(from_currency, to_currency, weight=-np.log(ask_price))
                
                # Edge: to_currency -> from_currency (sell to_currency, buy from_currency)
                # This is the inverse pair. Effective rate = 1 / bid_price
                G.add_edge(to_currency, from_currency, weight=-np.log(1 / bid_price))
        graphs[timestamp] = G
    return graphs


def detect_arbitrage_incremental(df):
    """
    New incremental detection using ArbitrageDetector class.
    Reuses graph topology and arrays across timestamps (O(VE) per timestamp).
    """
    df = df[["timestamp", "pair", "close"]]
    df_pivot = df.pivot(index="timestamp", columns="pair", values="close")
    
    currencies = ["USD", "EUR", "JPY", "GBP", "CNH", "AUD", "CAD", "CHF", "HKD", "SGD"]
    detector = ArbitrageDetector(currencies, SPREAD_FRACTION, MIN_PROFIT_MULTIPLIER)
    
    return detector.detect(df_pivot)

In [ ]:
def normalize_cycle(cycle):
    cycle = cycle[:-1]
    min_idx = min(range(len(cycle)), key=lambda i: cycle[i])
    normalized = cycle[min_idx:] + cycle[:min_idx]
    return tuple(normalized)

def find_all_negative_cycles(graph, min_profit=MIN_PROFIT_MULTIPLIER):
    all_cycles = set()
    cycles_info = []

    for source in graph.nodes:
        nodes = list(graph.nodes())
        distance = {node: float('inf') for node in nodes}
        predecessor = {node: None for node in nodes}
        distance[source] = 0
        # Bellman-Ford algorithm to find shortest paths and detect negative cycles
        for _ in range(len(nodes) - 1):
            for u, v, data in graph.edges(data=True):
                weight = data["weight"]
                if distance[u] + weight < distance[v]:
                    distance[v] = distance[u] + weight
                    predecessor[v] = u
        for u, v, data in graph.edges(data=True):
            weight = data["weight"]
            if distance[u] + weight < distance[v]:
                curr = v
                for _ in range(len(nodes)):
                    curr = predecessor.get(curr)
                    if curr is None:
                        break
                if curr is None:
                    continue

                cycle = []
                visited = set()
                start = curr
                while True:
                    if curr is None or curr in visited:
                        break
                    visited.add(curr)
                    cycle.append(curr)
                    curr = predecessor.get(curr)
                    if curr == start and len(cycle) > 1:
                        cycle.append(curr)
                        break

                if len(cycle) > 1 and curr == start:
                    cycle = cycle[::-1]
                    norm = normalize_cycle(cycle)
                    if norm not in all_cycles:
                        profit_log = sum(graph[u][v]["weight"] for u, v in zip(cycle, cycle[1:]))
                        profit = np.exp(-profit_log)
                        # Only keep cycles that exceed the minimum profit threshold
                        if profit >= min_profit:
                            all_cycles.add(norm)
                            cycles_info.append({
                                "cycle": " → ".join(cycle),
                                "log_sum": profit_log,
                                "profit_multiplier": profit
                            })
    return cycles_info

In [ ]:
output_folder = "Arbitrage_Cycles"
os.makedirs(output_folder, exist_ok=True)

# Use incremental detector for all days
for date, df in all_days_data.items():
    print(f"Processing {date}...")
    cycles = detect_arbitrage_incremental(df)
    for c in cycles:
        c["date"] = date
    df_day = pd.DataFrame(cycles)
    out_file = os.path.join(output_folder, f"arbitrage_{date}.csv")
    df_day.to_csv(out_file, index=False)
    print(f"{len(df_day)} arbitrage cycles (profit ≥ {MIN_PROFIT_MULTIPLIER:.4f}) saved to {out_file}")